### [ 데이터 기반 커스텀 모델 클래스 + 손실 계산 ]
- 부모 클래스 : nn.Module - 모든 인공신경망 클래스의 조상 (python - Object Class)
- 필수 오버라이딩 메서드
    * __init__(self) : 층 구성 요소들 초기화
    * forward(self, data) : 순전파 기능 메서드

- 손실계산용 함수 & 클래스
    * 함수 : torch.nn.functional => 어디서든 호출해서 사용
    * 클래스 : torch.nn => 인스턴스 생성해서 사용 => 하나의 층 매개변수 전달
    * **중요 : 모델의 출력층에 사용할 할성화 함수(AF) 선택에 영향을 미침**
        - 'logit' 단어 있는 함수 : 가중치 합계를 전달 
        - 'probability' 단어 있는 함수 : 확률 변환 AF 처리 후 전달

In [2]:
import torch
import torch.nn as nn # 인공신경망 관련 모듈
import torch.nn.functional as F #

> Senquential 클래스 이해하기
- 다른 모듈 (층, 활성화 함수 등등)을 여러개 담을 수 있는 클래스
- 담긴 순서대로 데이터를 전달 및 수행

#### 모델 구성 설계

| 층 | 입력 | 출력 | 활성화함수 |
|---|---|---|---|
| 입력층 | 피쳐5개 | 피쳐5개 | X |
| 은닉층 | 피쳐5개 | 피쳐10개 | AF(10개) |
| 은닉층 | 결과10개 | 피쳐7개 | AF(7개) |
| 출력층 | 결과7개 | 불류- 클래스 개수 회귀 1개 | AF(클래스 개수) 1개 |

In [3]:
# ----------------------------------------------------------
# [1] 분류 예시 - iris 품종 분류
# - 데이터 : 피쳐 4개 | 타겟 3개
# - 학습 종류 : 지도학습 + 분류
# - 입력층 : 입력 4개 | 출력층 : 출력 3개, 활성 함수 사용 Softmax
# - 은닉층 : 입력 전층 결과값, 출력 알아서..., 활성함수 ReLU
# ----------------------------------------------------------
# 구성 층 순서대로 입력
class IrisModel(nn.Module):
    # 모델 층 구성 요소 생성
    def __init__(self):
        super().__init__()
        self.hd1_layer = nn.Linear(4, 20)
        self.hd2_layer = nn.Linear(20, 30)
        self.hd3_layer = nn.Linear(30, 15)
        self.out_layer = nn.Linear(15, 3)

    # 순전파 진행 메서드
    def forward(self, data):
        out = self.hd1_layer(data)  # 가중합계 연산 수행 결과 반환
        out = F.relu(out)           # 활성화함수 거쳐서 값이 0이상으로 변환

        out = self.hd2_layer(out)   # 이전 은닉층 결과에 가중합계 연산 수행
        out = F.relu(out)           # 활성화함수 거쳐서 값이 0이상으로 변환

        out = self.hd3_layer(out)   # 이전 은닉층 결과에 가중합계 연산 수행
        out = F.relu(out)           # 활성화함수 거쳐서 값이 0이상으로 변환

        out = self.out_layer(out)   # 마지막 은닉층 결과에 가중합계 연산 수행
        out = F.softmax(out, dim=1) # 활성화함수 거쳐서 확율값으로 변환
        return out

In [30]:
# ----------------------------------------------------------
# [1] 분류 예시 - iris 품종 분류
# - 데이터 : 피쳐 4개 | 타겟 3개
# - 학습 종류 : 지도학습 + 분류
# - 입력층 : 입력 4개 | 출력층 : 출력 3개, 활성 함수 사용 Softmax
# - 은닉층 : 입력 전층 결과값, 출력 알아서..., 활성함수 ReLU
# ----------------------------------------------------------
# 구성 층 순서대로 입력
class MyClasfication(nn.Module):
    # 모델 층 구성 요소 생성
    def __init__(self, in_, out_, isbinary=True, return_proba=True):
        super().__init__()
        self.hd1_layer = nn.Linear(in_, 20)
        self.hd2_layer = nn.Linear(20, 30)
        self.hd3_layer = nn.Linear(30, 15)
        self.out_layer = nn.Linear(15, out_)
        self.isbinary = isbinary                # 분류 종류 True : 이진 분류, False : 다중분류
        self.return_proba = return_proba        # 출력층 반환값 종류 True : 확률값 반환, False : 가중값 반환

    # 순전파 진행 메서드
    def forward(self, data):
        out = F.relu(self.hd1_layer(data))
        out = F.relu(self.hd2_layer(out)) 
        out = F.relu(self.hd3_layer(out))
        out = self.out_layer(out)

        # 출력층의 활성화 함수는 학습종류와 손실함수에 따라 선택
        # 다중 분류로 softmax를 사용해야 하지만,
        # 손실함수가 내부적으로 확률계산 진행 함. 그래서 필요없음
        if self.isbinary:
            return F.sigmoid(out) if self.return_proba else out
        else:
            return F.softmax(out, dim=1) if self.return_proba else out

In [33]:
model = IrisModel()
model2 = MyClasfication(4, 3, False, False)
print(model, model2, sep='\n\n')

IrisModel(
  (hd1_layer): Linear(in_features=4, out_features=20, bias=True)
  (hd2_layer): Linear(in_features=20, out_features=30, bias=True)
  (hd3_layer): Linear(in_features=30, out_features=15, bias=True)
  (out_layer): Linear(in_features=15, out_features=3, bias=True)
)

MyClasfication(
  (hd1_layer): Linear(in_features=4, out_features=20, bias=True)
  (hd2_layer): Linear(in_features=20, out_features=30, bias=True)
  (hd3_layer): Linear(in_features=30, out_features=15, bias=True)
  (out_layer): Linear(in_features=15, out_features=3, bias=True)
)


In [34]:
# 모델 층별 파라미터 즉 w, b 초기값 확인
for names, parms in model2.named_parameters(): 
    print(f'[{names}]', parms.shape, end='\n\n')

[hd1_layer.weight] torch.Size([20, 4])

[hd1_layer.bias] torch.Size([20])

[hd2_layer.weight] torch.Size([30, 20])

[hd2_layer.bias] torch.Size([30])

[hd3_layer.weight] torch.Size([15, 30])

[hd3_layer.bias] torch.Size([15])

[out_layer.weight] torch.Size([3, 15])

[out_layer.bias] torch.Size([3])



In [38]:
# 순전파 진행 => Float32 타입
test = torch.FloatTensor([[1.2, 4.5, 3.3, 2.8]])
pre_y1 = model(test)
pre_y2 = model2(test)

print(f'{pre_y1}\n{pre_y2}')

tensor([[0.2631, 0.3482, 0.3887]], grad_fn=<SoftmaxBackward0>)
tensor([[0.0638, 0.1999, 0.1940]], grad_fn=<AddmmBackward0>)
